# Suite CONF — Conformance across implementations

Every implementation writes the same files for the conformance corpus (`conformance/`): each language's grammar,
and for each source the tree it parses to, as a JSON snapshot, and the text that tree prints to. This suite checks
that this implementation's files are current, that they are byte-identical to every other implementation's, and that
every implementation's snapshots read back into trees that print the same text.

In [ ]:
import json
from pathlib import Path

from mbse.Programs.Ccpp import Ccpp20, Syntax as S
from mbse.Programs.Conformance.write import render
from mbse.Programs.Framework import Syntax as F
from mbse.Schemas.Framework import JSON

ROOT = Path("../../conformance")
MINE = ROOT / "python3"
OTHERS = [d for d in sorted(ROOT.iterdir()) if d.is_dir() and d.name not in ("sources", MINE.name)]
files = render(ROOT / "sources")

## CONF-01 · This implementation's files are current

In [ ]:
assert sorted(p.name for p in MINE.iterdir()) == sorted(files), "run python -m mbse.Programs.Conformance.write"
for name, text in files.items():
    assert (MINE / name).read_text(encoding="utf-8") == text, f"{name} is stale"
grammar = json.loads(files["Ccpp.grammar.json"])
assert [k["kind"] for k in grammar["kinds"]] == list(S.LANGUAGE.kinds())
print(sorted(files))

## CONF-02 · Every other implementation wrote the same bytes

In [ ]:
assert [d.name for d in OTHERS] == ["typescript5"], OTHERS
for other in OTHERS:
    assert sorted(p.name for p in other.iterdir()) == sorted(files), other
    for name, text in files.items():
        assert (other / name).read_text(encoding="utf-8") == text, f"{other.name}/{name} differs"
print("identical to", [d.name for d in OTHERS])

## CONF-03 · Every snapshot reads back into a valid tree that prints the same text

In [ ]:
for directory in [MINE, *OTHERS]:
    for snapshot in sorted(directory.glob("*.json")):
        if snapshot.name.endswith(".grammar.json"):
            continue
        unit = JSON.FromJSON(S.LANGUAGE.Builders).Reachable(S.TranslationUnit.Schema, snapshot.read_text(encoding="utf-8"))
        assert S.LANGUAGE.validate(unit) == []
        printed = (directory / (snapshot.stem + ".cpp")).read_text(encoding="utf-8")
        assert Ccpp20.print(unit) == printed and F.same(Ccpp20.parse(printed), unit)
print("ok")